In [1]:
import pandas as pd

df = pd.read_csv('Dataset1.csv')
print("--top 5 rows--")
print(df.head(5))

--top 5 rows--
   SN  Train_No Station_Code   1A   2A   3A   SL  Station_Name  Route_Number  \
0   1       107          SWV  100  100  100  100  SAWANTWADI R             1   
1   2       107         THVM  260  228  196  164        THIVIM             1   
2   3       107         KRMI  345  296  247  198       KARMALI             1   
3   4       107          MAO  490  412  334  256   MADGOAN JN.             1   
4   1       108          MAO  100  100  100  100   MADGOAN JN.             1   

  Arrival_time Departure_Time  Distance  
0     00:00:00       10:25:00         0  
1     11:06:00       11:08:00        32  
2     11:28:00       11:30:00        49  
3     12:10:00       00:00:00        78  
4     00:00:00       20:30:00         0  


In [2]:
#Standardize Schedule Fields , arrival and departure time of train 
df['Arrival_time'] = df['Arrival_time'].astype(str).str.strip()
#arrival and departure time ki values ko string me convert karta hay(.astype) and .str.strip() values ke age piche ki blank space delete karta hay
df['Departure_Time'] = df['Departure_Time'].astype(str).str.strip()


In [3]:
df['Standard_arrival'] = pd.to_datetime(df['Arrival_time'], format= '%H:%M:%S', errors = 'coerce').dt.time
#std arrival varialble me arrival time ko std form me set karenga , agar koi error aata hay to vaha null store honga(errors = coerde ke karan)/ .dt.time se sirf time lenga
df['Standard_departure'] = pd.to_datetime(df['Departure_Time'], format = '%H:%M:%S', errors = 'coerce').dt.time

In [4]:
df[['Train_No', 'Station_Name', 'Arrival_time', 'Standard_arrival', 'Departure_Time', 'Standard_departure']].head()

,Train_No,Station_Name,Arrival_time,Standard_arrival,Departure_Time,Standard_departure
0,107,SAWANTWADI R,00:00:00,00:00:00,10:25:00,10:25:00
1,107,THIVIM,11:06:00,11:06:00,11:08:00,11:08:00
2,107,KARMALI,11:28:00,11:28:00,11:30:00,11:30:00
3,107,MADGOAN JN.,12:10:00,12:10:00,00:00:00,00:00:00
4,108,MADGOAN JN.,00:00:00,00:00:00,20:30:00,20:30:00


In [5]:
df_sorted = df.sort_values(by=['Train_No', 'SN'])

In [6]:
journey_summary = df_sorted.groupby('Train_No').agg(
    Start_Station=('Station_Name', 'first'),
    End_Station=('Station_Name', 'last'),
    Start_Dep=('Departure_Time', 'first'),
    End_Arr=('Arrival_time', 'last'),
    Total_Distance=('Distance', 'max')
).reset_index()

In [7]:
t_start = pd.to_timedelta(journey_summary['Start_Dep'].astype(str))
t_end = pd.to_timedelta(journey_summary['End_Arr'].astype(str))
duration = t_end - t_start

In [8]:
duration = duration.apply(lambda d: d + pd.Timedelta(days=1) if d.total_seconds() < 0 else d)

In [9]:
journey_summary['Journey_Duration'] = duration
journey_summary['Duration_Hours'] = (journey_summary['Journey_Duration'].dt.total_seconds() / 3600).round(2)

In [10]:
journey_summary.head(5)

,Train_No,Start_Station,End_Station,Start_Dep,End_Arr,Total_Distance,Journey_Duration,Duration_Hours
0,107,SAWANTWADI R,MADGOAN JN.,10:25:00,12:10:00,78,0 days 01:45:00,1.75
1,108,MADGOAN JN.,SAWANTWADI R,20:30:00,22:25:00,83,0 days 01:55:00,1.92
2,128,MADGOAN JN.,CHHATRAPATI,19:40:00,17:45:00,978,0 days 22:05:00,22.08
3,290,DELHI-SAFDAR,DELHI-SAFDAR,18:30:00,02:30:00,2694,0 days 08:00:00,8.00
4,401,AURANGABAD,VARANASI JN.,21:30:00,10:00:00,1618,0 days 12:30:00,12.50


In [11]:
journey_summary.sort_values(by= 'Duration_Hours', ascending= False).head(10)

,Train_No,Start_Station,End_Station,Start_Dep,End_Arr,Total_Distance,Journey_Duration,Duration_Hours
1723,15001,MUZAFFARPUR,DEHRA DUN,14:00:00,13:55:00,1122,0 days 23:55:00,23.92
1300,12949,PORBANDAR,SANTRAGACHI,08:10:00,08:05:00,2664,0 days 23:55:00,23.92
374,11103,JHANSI JN,BANDRA TERMI,16:50:00,16:45:00,1321,0 days 23:55:00,23.92
1128,12766,AMRAVATI,TIRUPATI,06:45:00,06:35:00,1305,0 days 23:50:00,23.83
888,12494,HAZRAT NIZAM,PUNE JN.,21:35:00,21:25:00,1501,0 days 23:50:00,23.83
252,8617,RANCHI,ANAND VIHAR,14:05:00,13:55:00,1365,0 days 23:50:00,23.83
1140,12778,KOCHUVELI,HUBLI JN.,12:50:00,12:40:00,1297,0 days 23:50:00,23.83
1139,12777,HUBLI JN.,KOCHUVELI,06:45:00,06:30:00,1295,0 days 23:45:00,23.75
406,11404,CHHATRAPATI,NAGPUR JN.(C,12:50:00,12:35:00,1026,0 days 23:45:00,23.75
2187,17214,NAGARSOL,NARASAPUR,11:15:00,11:00:00,1092,0 days 23:45:00,23.75


In [12]:
def categorize_route(distance):
    if distance < 300:
        return 'Short'
    elif 300 <= distance <= 800:
        return 'Medium'
    else:
        return 'Long'

In [13]:
journey_summary['Route_Category'] = journey_summary['Total_Distance'].apply(categorize_route)

In [14]:
print(journey_summary['Route_Category'].value_counts())

Route_Category
Short     8221
Long      1546
Medium    1346
Name: count, dtype: int64


In [15]:
station_freq = df.groupby('Station_Name')['Train_No'].nunique().reset_index()

In [16]:
station_freq.rename(columns={'Train_No': 'Train_Frequency'}, inplace=True)
station_freq = station_freq.sort_values(by='Train_Frequency', ascending=False).reset_index(drop=True)

In [17]:
station_freq.head(10)

,Station_Name,Train_Frequency
0,CST-MUMBAI,1027
1,KALYAN JN,828
2,THANE,796
3,SEALDAH,745
4,CHENNAI BEAC,738
5,HOWRAH JN.,699
6,DADAR,598
7,DUM DUM JN.,463
8,KURLA,462
9,TAMBARAM,434
